In [ ]:
import duckdb
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import date
import textstat
from transformers import pipeline

In [ ]:
con = duckdb.connect('../../ProjectData.duckdb')
con.execute('CREATE SCHEMA IF NOT EXISTS gold')

silver_files = sorted(Path('../../data/silver').glob('cleaned_data_load_date=*.parquet'))
if not silver_files:
    raise FileNotFoundError('No Silver Parquet found. Run clean_transform_to_silver.ipynb first.')

silver_path = silver_files[-1]
print(f'Loading Silver from: {silver_path}')

silver_df = pd.read_parquet(silver_path)
print(f'Rows loaded:  {len(silver_df):,}')
print(f'Columns:      {list(silver_df.columns)}')

In [ ]:
SENTIMENT_MODEL = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
print(f'Loading sentiment model: {SENTIMENT_MODEL} ...')
sentiment_pipe = pipeline(
    'text-classification',
    model=SENTIMENT_MODEL,
    top_k=None,
    truncation=True,
    max_length=256,
    device=-1,
)
print('Model loaded.')

In [ ]:
def score_sentiment(texts, batch_size=32):
    '''
    Encode texts with the multilingual sentiment model.
    Returns DataFrame: label, score, polarity (pos_score - neg_score in [-1,1]).
    Null/empty inputs receive None values.
    '''
    clean = [str(t).strip() if t and str(t).strip() else ' ' for t in texts]
    valid = [t != ' ' for t in clean]
    results = sentiment_pipe(clean, batch_size=batch_size)
    rows = []
    for res, is_valid in zip(results, valid):
        if not is_valid:
            rows.append({'label': None, 'score': None, 'polarity': None})
            continue
        scores = {r['label']: r['score'] for r in res}
        pos = scores.get('Positive', 0.0)
        neg = scores.get('Negative', 0.0)
        top = max(res, key=lambda x: x['score'])
        rows.append({'label': top['label'], 'score': top['score'], 'polarity': float(pos - neg)})
    return pd.DataFrame(rows)

print('Encoding review body sentiment (~2-5 min on CPU) ...')
body_sent = score_sentiment(silver_df['review_body'].tolist())
silver_df['body_sentiment_label']    = body_sent['label'].values
silver_df['body_sentiment_score']    = body_sent['score'].values
silver_df['body_sentiment_polarity'] = body_sent['polarity'].values

print('Encoding headline sentiment ...')
headline_sent = score_sentiment(silver_df['review_headline'].tolist())
silver_df['headline_sentiment_label']    = headline_sent['label'].values
silver_df['headline_sentiment_score']    = headline_sent['score'].values
silver_df['headline_sentiment_polarity'] = headline_sent['polarity'].values

silver_df['sentiment_mismatch'] = np.where(
    silver_df['body_sentiment_polarity'].isna() | silver_df['headline_sentiment_polarity'].isna(),
    np.nan,
    np.abs(silver_df['body_sentiment_polarity'] - silver_df['headline_sentiment_polarity'])
)

print('Body sentiment distribution:')
print(silver_df['body_sentiment_label'].value_counts())

In [ ]:
def repetition_ratio(text):
    '''Fraction of sentences repeated verbatim (lowercased, stripped). Returns 0.0 for single-sentence texts.'''
    if not text or not str(text).strip():
        return None
    sentences = re.split(r'[.!?]+', str(text))
    sentences = [s.strip().lower() for s in sentences if s.strip()]
    if len(sentences) <= 1:
        return 0.0
    total = len(sentences)
    unique = len(set(sentences))
    return (total - unique) / total

TEXTSTAT_LANG_MAP = {
    'en': 'en', 'de': 'de', 'fr': 'fr', 'es': 'es',
    'it': 'it', 'nl': 'nl', 'ru': 'ru'
}

def flesch_ease(row):
    '''Flesch reading ease, language-aware. Falls back to English syllable counting for unknown languages.'''
    lang = TEXTSTAT_LANG_MAP.get(row.get('detected_language', ''), 'en')
    text = row.get('review_body', '') or ''
    if not text.strip():
        return None
    textstat.set_lang(lang)
    try:
        return float(textstat.flesch_reading_ease(text))
    except Exception:
        return None

try:
    lang_df = con.execute(
        'SELECT _index, detected_language FROM gold.review_lexical_features'
    ).df()
    silver_df = silver_df.merge(lang_df, on='_index', how='left')
    print('Loaded detected_language from gold.review_lexical_features.')
except Exception:
    from langdetect import detect, LangDetectException, DetectorFactory
    DetectorFactory.seed = 0
    def detect_language(text):
        if not text or len(str(text).strip()) < 10:
            return 'unknown'
        try:
            return detect(str(text))
        except LangDetectException:
            return 'unknown'
    silver_df['detected_language'] = silver_df['review_body'].apply(detect_language)
    print('Re-computed detected_language via langdetect.')

print('Computing repetition ratio ...')
silver_df['repetition_ratio'] = silver_df['review_body'].apply(repetition_ratio)

print('Computing Flesch reading ease ...')
silver_df['flesch_reading_ease'] = silver_df[['review_body', 'detected_language']].apply(flesch_ease, axis=1)

print(silver_df[['repetition_ratio', 'flesch_reading_ease']].describe().round(3))

In [ ]:
def jaccard_sim(text_a, text_b):
    '''Token-level Jaccard similarity: |intersection| / |union|.'''
    if not text_a or not text_b:
        return None
    a = set(str(text_a).lower().split())
    b = set(str(text_b).lower().split())
    union = len(a | b)
    return len(a & b) / union if union > 0 else 0.0

def overlap_ratio(text_source, text_target):
    '''Asymmetric overlap: fraction of source tokens present in target.'''
    if not text_source or not text_target:
        return None
    src = set(str(text_source).lower().split())
    tgt = set(str(text_target).lower().split())
    return len(src & tgt) / len(src) if src else None

def bigram_diversity(text):
    '''Unique bigrams / total bigrams: phrasal richness beyond type-token ratio.'''
    if not text or not str(text).strip():
        return None
    tokens = str(text).lower().split()
    if len(tokens) < 2:
        return None
    bigrams = list(zip(tokens, tokens[1:]))
    return len(set(bigrams)) / len(bigrams)

print('Computing lexical overlap features ...')
silver_df['headline_body_jaccard'] = silver_df.apply(
    lambda r: jaccard_sim(r['review_headline'], r['review_body']), axis=1)
silver_df['title_body_jaccard'] = silver_df.apply(
    lambda r: jaccard_sim(r['product_title'], r['review_body']), axis=1)
silver_df['title_body_overlap'] = silver_df.apply(
    lambda r: overlap_ratio(r['product_title'], r['review_body']), axis=1)
silver_df['body_bigram_diversity'] = silver_df['review_body'].apply(bigram_diversity)

print(silver_df[[
    'headline_body_jaccard', 'title_body_jaccard',
    'title_body_overlap', 'body_bigram_diversity'
]].describe().round(3))

In [ ]:
con.register('silver_enriched', silver_df)

create_sql = '''
    CREATE TABLE gold.review_sentiment_features AS
    WITH base AS (
        SELECT
            _index,
            product_id,
            product_parent,
            label,
            detected_language,
            product_category_id,
            body_sentiment_label,
            body_sentiment_score,
            body_sentiment_polarity,
            headline_sentiment_label,
            headline_sentiment_score,
            headline_sentiment_polarity,
            sentiment_mismatch,
            repetition_ratio,
            flesch_reading_ease,
            headline_body_jaccard,
            title_body_jaccard,
            title_body_overlap,
            body_bigram_diversity,
            REGEXP_COUNT(review_body, '[A-Z]') * 1.0
                / NULLIF(REGEXP_COUNT(review_body, '[a-zA-Z]'), 0)
                AS uppercase_ratio,
            REGEXP_COUNT(review_body, '[0-9]') * 1.0
                / NULLIF(ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')), 0)
                AS digit_density,
            GREATEST(REGEXP_COUNT(review_body, '[.!?]+'), 1) AS sentence_count,
            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')) * 1.0
                / NULLIF(GREATEST(REGEXP_COUNT(review_body, '[.!?]+'), 1), 0)
                AS avg_sentence_length
        FROM silver_enriched
        WHERE review_body IS NOT NULL AND TRIM(review_body) != ''
    ),
    normed AS (
        SELECT
            *,
            (flesch_reading_ease
                - AVG(flesch_reading_ease) OVER (PARTITION BY detected_language))
                / NULLIF(STDDEV(flesch_reading_ease) OVER (PARTITION BY detected_language), 0)
                AS flesch_lang_zscore,
            (flesch_reading_ease
                - AVG(flesch_reading_ease) OVER (PARTITION BY product_category_id))
                / NULLIF(STDDEV(flesch_reading_ease) OVER (PARTITION BY product_category_id), 0)
                AS flesch_cat_zscore,
            (body_sentiment_polarity
                - AVG(body_sentiment_polarity) OVER (PARTITION BY detected_language))
                / NULLIF(STDDEV(body_sentiment_polarity) OVER (PARTITION BY detected_language), 0)
                AS polarity_lang_zscore
        FROM base
    )
    SELECT * FROM normed
'''

con.execute('DROP TABLE IF EXISTS gold.review_sentiment_features')
con.execute(create_sql)
print('gold.review_sentiment_features created.')
print('Rows:', con.execute('SELECT COUNT(*) FROM gold.review_sentiment_features').fetchone()[0])

In [ ]:
con.execute('''
    SELECT
        COUNT(*)                                                              AS total_rows,
        SUM(CASE WHEN body_sentiment_label IS NULL THEN 1 ELSE 0 END)        AS null_body_sentiment,
        SUM(CASE WHEN headline_sentiment_label IS NULL THEN 1 ELSE 0 END)    AS null_headline_sentiment,
        SUM(CASE WHEN flesch_reading_ease IS NULL THEN 1 ELSE 0 END)         AS null_flesch,
        SUM(CASE WHEN repetition_ratio IS NULL THEN 1 ELSE 0 END)            AS null_repetition,
        ROUND(MIN(body_sentiment_polarity), 3)                               AS min_polarity,
        ROUND(MAX(body_sentiment_polarity), 3)                               AS max_polarity,
        ROUND(AVG(body_sentiment_polarity), 3)                               AS avg_polarity,
        ROUND(AVG(sentiment_mismatch), 3)                                    AS avg_mismatch,
        ROUND(AVG(uppercase_ratio), 4)                                       AS avg_uppercase_ratio,
        ROUND(AVG(digit_density), 4)                                         AS avg_digit_density,
        ROUND(AVG(sentence_count), 1)                                        AS avg_sentence_count,
        ROUND(AVG(avg_sentence_length), 1)                                   AS avg_sentence_length,
        ROUND(AVG(headline_body_jaccard), 3)                                 AS avg_hl_body_jaccard,
        ROUND(AVG(body_bigram_diversity), 3)                                 AS avg_bigram_diversity
    FROM gold.review_sentiment_features
''').df()

In [ ]:
con.execute('''
    SELECT
        label,
        COUNT(*)                                         AS reviews,
        ROUND(AVG(body_sentiment_polarity), 3)           AS avg_polarity,
        ROUND(AVG(sentiment_mismatch), 3)                AS avg_mismatch,
        ROUND(AVG(polarity_lang_zscore), 3)              AS avg_polarity_zscore,
        ROUND(AVG(flesch_reading_ease), 2)               AS avg_flesch,
        ROUND(AVG(flesch_lang_zscore), 3)                AS avg_flesch_zscore,
        ROUND(AVG(repetition_ratio), 4)                  AS avg_repetition,
        ROUND(AVG(uppercase_ratio), 4)                   AS avg_uppercase,
        ROUND(AVG(digit_density), 4)                     AS avg_digit_density,
        ROUND(AVG(sentence_count), 1)                    AS avg_sentence_count,
        ROUND(AVG(avg_sentence_length), 1)               AS avg_sentence_length,
        ROUND(AVG(headline_body_jaccard), 3)             AS avg_hl_body_jaccard,
        ROUND(AVG(title_body_jaccard), 3)                AS avg_title_body_jaccard,
        ROUND(AVG(title_body_overlap), 3)                AS avg_title_overlap,
        ROUND(AVG(body_bigram_diversity), 3)             AS avg_bigram_diversity
    FROM gold.review_sentiment_features
    WHERE label IS NOT NULL
    GROUP BY label
    ORDER BY label DESC
''').df()

In [ ]:
gold_dir = Path('../../data/gold')
gold_dir.mkdir(parents=True, exist_ok=True)

today = date.today().isoformat()
output_file = gold_dir / f'sentiment_features_load_date={today}.parquet'

con.execute(f'''
    COPY (SELECT * FROM gold.review_sentiment_features)
    TO '{output_file.as_posix()}' (FORMAT PARQUET)
''')
print(f'Saved: {output_file.resolve()}')

In [ ]:
con.close()